# Bilder holen

Laedt fuer jedes Bild aus `metadata.parquet` das Mapillary-Thumbnail
nach `image_root/<stadt>` (`src/paths.py`). Vorhandene Dateien werden
uebersprungen, ein abgebrochener Lauf also einfach fortgesetzt.

Was der Durchsatz braucht, steht in `config.yaml`:

- `download_workers` — parallele Downloads (`"auto"` = 96)
- `download_image_size` — Breite der Thumbnails (1024)
- `max_missing_images_frac` — wie viele Bilder fehlen duerfen, bevor
  dieses Notebook abbricht statt einen unvollstaendigen Bestand als
  fertig auszugeben
- `verify_all_images` — `false` prueft nur die in diesem Lauf
  geholten Bilder, `true` den gesamten Bestand

Das Token kommt aus `.env` (`MAPILLARY_TOKEN`), nie aus `config.yaml`.
Die Geschwindigkeit haengt an der Netzanbindung, nicht an der CPU.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm


PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, paths
from src.mapillary import bild_ist_heil, get_session, load_token

CFG = load_config(PROJECT_ROOT)
PATHS = paths(CFG, PROJECT_ROOT)
PROCESSED_DIR = PATHS.processed
DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
IMAGE_PATH = PATHS.images
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

# Daneben der Ordner fuer eigene Fotos -- locate.py und die Demo lesen ihn.
OWN_PATH = PATHS.own_images
OWN_PATH.mkdir(parents=True, exist_ok=True)
(OWN_PATH / "README.txt").write_text(
    "Eigene Fotos hier ablegen (jpg/png). Verorten mit\n"
    "  python locate.py <bild>            oder den ganzen Ordner:\n"
    "  python locate.py " + str(OWN_PATH) + "\n"
    "Oder in demo/demo.ipynb: wo_ist_das(<bild>)\n"
, encoding="utf-8")
IMG_SIZE = CFG["download_image_size"]

# Schranke und Pruefumfang stehen in der config, nicht im Notebook.
MAX_MISSING_FRAC = float(CFG["max_missing_images_frac"])
PRUEFE_ALLE = bool(CFG.get("verify_all_images", False))
DOWNLOAD_STATE_PATH = PROCESSED_DIR / "image_download.json"
metadata = pd.read_parquet(DATA_PATH_META)
download_metadata = metadata[metadata["split"].isin(["train","database", "query"])].copy()

# Set Max Workers -> Recommendation for new Laptop 96
MAX_WORKERS = CFG["download_workers"]
if MAX_WORKERS == "auto":
    MAX_WORKERS = 96
else:
    MAX_WORKERS = int(MAX_WORKERS)


TOKEN = load_token(PROJECT_ROOT)


print("Download Location: ", IMAGE_PATH)
print(f"Download Threads: {MAX_WORKERS}")
print(f"Zu ladende Bilder: {len(download_metadata):,}")

# API Anfrage
  
Erstelle session, frage API an 

In [ ]:
def download_image(image_id):

    image_id = str(image_id)
    image_file = IMAGE_PATH / f"{image_id}.jpg"

    # Bereits vorhandenes, nicht-leeres Bild überspringen
    if image_file.exists() and image_file.stat().st_size > 0:
        return "exists"

    session = get_session(MAX_WORKERS)

    try:
        api_url = f"https://graph.mapillary.com/{image_id}"

        params = {
            "fields": f"id,thumb_{IMG_SIZE}_url",
            "access_token": TOKEN,
        }

        api_response = session.get(
            api_url,
            params=params,
            timeout=(10, 30),
        )

        api_response.raise_for_status()

        data = api_response.json()

        image_url = data.get(f"thumb_{IMG_SIZE}_url")

        if not image_url:
            print(f"Keine Bild-URL für {image_id}")
            return "failed"

        image_response = session.get(image_url, timeout=(10, 60))
        image_response.raise_for_status()

        if not image_response.content:
            print(f"Leere Bildantwort für {image_id}")
            return "failed"

        # Erst daneben schreiben, dann umbenennen. Ein Strg-C oder ein
        # OOM mitten im Schreiben hinterliess sonst eine JPG-Datei mit
        # Groesse > 0: der naechste Lauf haelt sie fuer vorhanden und
        # ueberspringt sie, und die Pruefung unten sieht nur die in
        # DIESEM Lauf geholten Bilder -- das halbe Bild wird also nie
        # geprueft. Path.replace ist auf einem Dateisystem atomar.
        teil = image_file.with_suffix(".jpg.part")
        teil.write_bytes(image_response.content)
        teil.replace(image_file)

        return "downloaded"

    except requests.RequestException as e:
        status = e.response.status_code if e.response is not None else "keine Antwort"

        print(f"Fehler bei {image_id}: {type(e).__name__}: {status}")

        return "failed"

    except (ValueError, KeyError) as e:
        print(f"Ungültige Mapillary-Antwort für {image_id}: {type(e).__name__}")

        return "failed"

    except Exception as e:
        # Fängt unerwartete Fehler ab, damit ein einzelnes Bild
        # nicht den gesamten 98k-Download stoppt.
        print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")

        return "failed"


# Download

In [ ]:

# Ein Verzeichnisdurchlauf statt 332.868 einzelner Abfragen im Threadpool:
# beim wiederholten Lauf ist praktisch alles schon da.
# Liegengebliebene .part-Dateien aus einem abgebrochenen Lauf zuerst weg --
# sie sind per Definition unvollstaendig.
for rest in IMAGE_PATH.glob("*.jpg.part"):
    rest.unlink()

vorhandene_ids = {
    eintrag.name[:-4]
    for eintrag in os.scandir(IMAGE_PATH)
    if eintrag.name.endswith(".jpg") and eintrag.stat().st_size > 0
}

alle_ids = [str(i) for i in download_metadata["image_id"]]
fehlende_ids = [i for i in alle_ids if i not in vorhandene_ids]

print(f"Bereits vorhanden: {len(alle_ids) - len(fehlende_ids):,}")
print(f"Zu holen:          {len(fehlende_ids):,}")

results = {}

if fehlende_ids:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(download_image, image_id): image_id
            for image_id in fehlende_ids
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc="Bilder herunterladen"):
            image_id = futures[future]
            try:
                results[image_id] = future.result()
            except Exception as e:
                print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")
                results[image_id] = "failed"

# Nur diese muessen anschliessend auf Unversehrtheit geprueft werden.
neu_geladene_ids = [i for i, r in results.items() if r == "downloaded"]
failed_ids = [i for i, r in results.items() if r == "failed"]
image_files = list(IMAGE_PATH.glob("*.jpg"))
FAILED_IMAGE_PATH = PROCESSED_DIR / "failed_image_download.txt"
FAILED_IMAGE_PATH.write_text("\n".join(map(str, failed_ids)), encoding="utf-8")


# Auswertung erster Download
print("=" * 50)
print("DOWNLOAD ERGEBNISSE")
print("=" * 50)
print(f"Neu geladen:                {len(neu_geladene_ids):,}")
print(f"Bereits vorhanden:          {len(alle_ids) - len(fehlende_ids):,}")
print(f"Fehlgeschlagene:            {list(results.values()).count('failed'):,}")
print("=" * 50)
print(f"Fehlgeschlagene Bilder:     {len(failed_ids):,}")
print(f"Liste gespeichert unter:    {FAILED_IMAGE_PATH}")
print(f"Lokale JPGs:                {len(image_files):,}")
print("=" * 50)

# Was wirklich auf der Platte liegt -- nicht nur, was dieser Lauf geholt hat.
# Ein abgelaufener Token laesst sonst eine Fehlerliste und einen "fertigen"
# Zustand zurueck, und 04 encodiert stillschweigend den Rest.
vorhanden_jetzt = {
    eintrag.name[:-4]
    for eintrag in os.scandir(IMAGE_PATH)
    if eintrag.name.endswith(".jpg") and eintrag.stat().st_size > 0
}
fehlend = [i for i in alle_ids if i not in vorhanden_jetzt]
anteil_fehlend = len(fehlend) / max(len(alle_ids), 1)

import json as _json
DOWNLOAD_STATE_PATH.write_text(_json.dumps({
    "erwartet": len(alle_ids),
    "vorhanden": len(alle_ids) - len(fehlend),
    "fehlend": len(fehlend),
    "anteil_fehlend": round(anteil_fehlend, 6),
    "max_missing_images_frac": MAX_MISSING_FRAC,
}, indent=2), encoding="utf-8")

print(f"Fehlend insgesamt:          {len(fehlend):,} ({anteil_fehlend:.3%})")
print(f"Zustand gespeichert unter:  {DOWNLOAD_STATE_PATH}")

if anteil_fehlend > MAX_MISSING_FRAC:
    raise RuntimeError(
        f"{len(fehlend):,} von {len(alle_ids):,} Bildern fehlen "
        f"({anteil_fehlend:.2%}) -- erlaubt sind {MAX_MISSING_FRAC:.2%} "
        "(config.yaml -> max_missing_images_frac).\n"
        "Meist ist der Mapillary-Token abgelaufen oder die Verbindung brach ab. "
        "Token pruefen und diese Zelle erneut ausfuehren -- vorhandene Bilder "
        "werden uebersprungen. Ist der Verlust echt (Mapillary hat Bilder "
        "entfernt), die Schranke bewusst hochsetzen."
    )


# Kaputte Bilder

In [ ]:
# Abgebrochene Downloads erkennen. Nur die Bilder aus diesem Lauf -- an den
# uebrigen kann sich nichts geaendert haben. verify_all_images: true in der
# config.yaml liest den gesamten Bestand, was bei 330k Dateien etliche
# Minuten dauert.

if PRUEFE_ALLE:
    zu_pruefen = list(IMAGE_PATH.glob("*.jpg"))
else:
    zu_pruefen = [IMAGE_PATH / f"{i}.jpg" for i in neu_geladene_ids]

# bild_ist_heil() dekodiert das Bild (load), statt nur den Kopf zu lesen
# (verify). Der Unterschied ist genau dieser Fall: ein bei 50 % oder 90 %
# abgeschnittenes JPEG haelt verify() fuer heil -- und abgeschnitten ist,
# was ein abgebrochener Download hinterlaesst.
bad = [img for img in tqdm(zu_pruefen, desc="Bilder pruefen")
       if not bild_ist_heil(img)]

print(f"Geprueft: {len(zu_pruefen):,}   kaputt: {len(bad)}")

if bad:
    for img in bad:
        img.unlink()
    print(
        f"{len(bad)} geloescht -- die Download-Zelle erneut ausfuehren, "
        "dann werden sie neu geholt."
    )
